# 1. Train and Evaluate MLCQ Code Smell Classifier

This notebook fine-tunes a lightweight language model for **binary per-smell code smell detection** on the prepared MLCQ dataset.

Design choices:

- Use **PEFT/LoRA** for parameter-efficient fine-tuning.
- Use **weighted cross-entropy loss** to handle class imbalance.
- Do **not** oversample.
- Keep model DataFrames minimal: only `text` and `label_id`.
- Load only `target_smell` and `label_id` separately for per-smell evaluation.

# 2. Install Dependencies

Run this cell in Kaggle. `peft` is required because this project trains with LoRA adapters.


In [55]:
!pip install -q transformers datasets accelerate evaluate scikit-learn peft


# 3. Imports

In [56]:
from pathlib import Path
import json
import random
import inspect
import math
import os
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorWithPadding,
)

from peft import LoraConfig, get_peft_model, TaskType


# 4. Configuration

In [57]:
SEED = 42

# Lightweight model for Kaggle-friendly PEFT training.
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Read prepared train/validation/test CSV files from this directory.
# If your prepared data is a Kaggle input dataset, change this to that read-only input path.
DATA_DIR = Path("/kaggle/input/datasets/fatemakotb02/mlcq-cleaned-binary-20")

# Save all training outputs to /kaggle/working because /kaggle/input is read-only.
OUTPUT_DIR = Path("/kaggle/working/outputs/training")
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model"
METRICS_OUTPUT_DIR = OUTPUT_DIR / "metrics"
PREDICTIONS_OUTPUT_DIR = OUTPUT_DIR / "predictions"
LOGGING_OUTPUT_DIR = OUTPUT_DIR / "logs"

for path in [OUTPUT_DIR, MODEL_OUTPUT_DIR, METRICS_OUTPUT_DIR, PREDICTIONS_OUTPUT_DIR, LOGGING_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Token length. Increase only if GPU memory allows.
MAX_LENGTH = 512

# Kaggle-friendly LoRA training settings.
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "linear"
MAX_GRAD_NORM = 1.0
SAVE_TOTAL_LIMIT = 1
LOGGING_STEPS = 20

# Debug mode: use a small subset to verify the full pipeline end to end.
# For a smoke test, set DEBUG_RUN = True. For the real experiment, keep it False.
DEBUG_RUN = False
DEBUG_N = 10

# PEFT/LoRA is required for this project.
USE_LORA = True

# LoRA settings.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

ID_TO_LABEL = {
    0: "absent",
    1: "present",
}

LABEL_TO_ID = {
    "absent": 0,
    "present": 1,
}

POSITIVE_LABEL_ID = 1


# 5. Set Random Seed

In [58]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


# 6. Load Only Needed Columns

In [59]:
train_path = DATA_DIR / "train.csv"
val_path = DATA_DIR / "val.csv"
test_path = DATA_DIR / "test.csv"

for path in [train_path, val_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset file: {path}")

# Load only the two columns needed by the model.
# This prevents the extra metadata columns from staying in memory.
model_columns = ["text", "label_id"]

train_df = pd.read_csv(train_path, usecols=model_columns)
val_df = pd.read_csv(val_path, usecols=model_columns)
test_df = pd.read_csv(test_path, usecols=model_columns)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (9383, 2)
Validation: (2034, 2)
Test: (2027, 2)


,label_id,text
0,0,You are a code smell detection model.\n\nTask:...
1,1,You are a code smell detection model.\n\nTask:...
2,1,You are a code smell detection model.\n\nTask:...
3,0,You are a code smell detection model.\n\nTask:...
4,0,You are a code smell detection model.\n\nTask:...


# 7. Load Minimal Metadata for Per-Smell Evaluation

In [60]:
# Load only the metadata needed for per-smell test metrics.
# Do not load the full 20-column file into memory.
test_meta_columns = ["target_smell", "label_id"]

test_meta_df = pd.read_csv(test_path, usecols=test_meta_columns)

if len(test_meta_df) != len(test_df):
    raise ValueError("test_meta_df and test_df have different row counts.")

print("Test metadata:", test_meta_df.shape)
test_meta_df.head()

Test metadata: (2027, 2)


,target_smell,label_id
0,feature envy,0
1,long method,0
2,feature envy,0
3,data class,0
4,data class,0


# 8. Apply Debug Subset

In [61]:
from sklearn.model_selection import train_test_split


def stratified_debug_indices(df: pd.DataFrame, n: int, seed: int):
    """Return row indices for a small label-stratified debug subset."""
    if len(df) <= n:
        return df.index.tolist()

    sampled_indices, _ = train_test_split(
        df.index,
        train_size=n,
        stratify=df["label_id"],
        random_state=seed,
    )

    return sorted(sampled_indices)


if DEBUG_RUN:
    train_idx = stratified_debug_indices(train_df, DEBUG_N, SEED)
    val_idx = stratified_debug_indices(val_df, DEBUG_N, SEED)
    test_idx = stratified_debug_indices(test_df, DEBUG_N, SEED)

    train_df = train_df.loc[train_idx].reset_index(drop=True)
    val_df = val_df.loc[val_idx].reset_index(drop=True)
    test_df = test_df.loc[test_idx].reset_index(drop=True)

    # Keep metadata aligned with the debug test rows.
    test_meta_df = test_meta_df.loc[test_idx].reset_index(drop=True)

    print("DEBUG RUN ENABLED")
    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)
    print("Test metadata:", test_meta_df.shape)

    print("\nTrain label counts:")
    print(train_df["label_id"].value_counts().sort_index())

    print("\nValidation label counts:")
    print(val_df["label_id"].value_counts().sort_index())

    print("\nTest label counts:")
    print(test_df["label_id"].value_counts().sort_index())

    print("\nTest smell counts:")
    print(test_meta_df["target_smell"].value_counts())


# 8. Dataset Sanity Checks

In [62]:
def check_model_dataframe(df: pd.DataFrame, name: str) -> None:
    required_columns = {"text", "label_id"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

    if df["text"].isna().any():
        raise ValueError(f"{name} has missing text values.")

    if not df["label_id"].isin([0, 1]).all():
        raise ValueError(f"{name} has invalid label_id values.")

    labels = set(df["label_id"].unique())
    if labels != {0, 1}:
        raise ValueError(f"{name} does not contain both labels. Found: {labels}")

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Label counts:")
    print(df["label_id"].value_counts().sort_index())


check_model_dataframe(train_df, "train")
check_model_dataframe(val_df, "validation")
check_model_dataframe(test_df, "test")


train
Rows: 9383
Label counts:
label_id
0    7332
1    2051
Name: count, dtype: int64

validation
Rows: 2034
Label counts:
label_id
0    1580
1     454
Name: count, dtype: int64

test
Rows: 2027
Label counts:
label_id
0    1565
1     462
Name: count, dtype: int64


# 9. Compute Class Weights

In [63]:
# Compute class weights from the training split only.
# Weighted loss handles class imbalance without changing the dataset distribution.
#
# Formula:
#   class_weight = total_training_examples / (number_of_classes * class_count)

label_counts = train_df["label_id"].value_counts().sort_index()

num_classes = 2
total_samples = len(train_df)

class_weights = []

for class_id in range(num_classes):
    class_count = label_counts.get(class_id, 0)

    if class_count == 0:
        raise ValueError(f"Class {class_id} does not appear in the training data.")

    class_weight = total_samples / (num_classes * class_count)
    class_weights.append(class_weight)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Training label counts:")
print(label_counts)

print("\nClass weights:")
print({
    "absent_weight": float(class_weights[0]),
    "present_weight": float(class_weights[1]),
})

Training label counts:
label_id
0    7332
1    2051
Name: count, dtype: int64

Class weights:
{'absent_weight': 0.6398663520812988, 'present_weight': 2.2874207496643066}


# 10. Load Tokenizer

In [64]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("lama")
print("Token loaded:", HF_TOKEN is not None, "length:", len(HF_TOKEN) if HF_TOKEN else 0)

Token loaded: True length: 37


In [65]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

models = api.list_models(author="meta-llama", search="Instruct", limit=200)
for m in models:
    print(m.modelId)

meta-llama/Llama-3.1-8B-Instruct
meta-llama/Meta-Llama-3-8B-Instruct
meta-llama/Llama-3.2-3B-Instruct
meta-llama/Llama-3.2-1B-Instruct
meta-llama/Llama-3.3-70B-Instruct
meta-llama/Llama-3.2-90B-Vision-Instruct
meta-llama/Llama-3.2-11B-Vision-Instruct
meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8
meta-llama/Llama-4-Maverick-17B-128E-Instruct
meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8-Original
meta-llama/CodeLlama-7b-Instruct-hf
meta-llama/CodeLlama-13b-Instruct-hf
meta-llama/CodeLlama-70b-Instruct-hf
meta-llama/CodeLlama-34b-Instruct-hf
meta-llama/Meta-Llama-3-70B-Instruct
meta-llama/Llama-3.1-70B-Instruct
meta-llama/Llama-3.1-405B-Instruct
meta-llama/Llama-3.1-405B-Instruct-FP8
meta-llama/Llama-3.2-1B-Instruct-QLORA_INT4_EO8
meta-llama/Llama-3.2-1B-Instruct-SpinQuant_INT4_EO8
meta-llama/Llama-3.2-3B-Instruct-QLORA_INT4_EO8
meta-llama/Llama-3.2-3B-Instruct-SpinQuant_INT4_EO8
meta-llama/Llama-4-Scout-17B-16E-Instruct
meta-llama/Llama-4-Scout-17B-16E-Instruct-Original
meta-llam

In [66]:
print(api.model_info(MODEL_NAME).modelId)

meta-llama/Llama-3.2-1B-Instruct


In [67]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Decoder-only models often do not define a pad token.
# For classification, using EOS as PAD is common.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

Pad token: <|eot_id|>
Pad token id: 128009


# 11. Convert to Hugging Face Datasets

In [68]:
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

print(train_dataset)

Dataset({
    features: ['labels', 'text'],
    num_rows: 9383
})


# 12. Tokenize Dataset

In [69]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

tokenized_val = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_train)


Map:   0%|          | 0/9383 [00:00<?, ? examples/s]

Map:   0%|          | 0/2034 [00:00<?, ? examples/s]

Map:   0%|          | 0/2027 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9383
})


# 13. Load Model

In [70]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM

class CausalLMForSequenceClassification(nn.Module):
    def __init__(self, base_lm, num_labels=2, pad_token_id=None):
        super().__init__()
        self.base_lm = base_lm
        self.model = base_lm.model          # helps PEFT find q_proj/v_proj
        self.config = base_lm.config
        self.num_labels = num_labels

        if not hasattr(self.config, "hidden_size"):
            raise ValueError(f"Config {type(self.config)} has no hidden_size attribute.")
        hidden_size = self.config.hidden_size

        self.score = nn.Linear(hidden_size, num_labels)

        self.config.pad_token_id = pad_token_id
        self.config.use_cache = False

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        hidden_states = outputs.last_hidden_state  # [B, T, H]

        if attention_mask is None:
            pooled = hidden_states[:, -1, :]
        else:
            lengths = attention_mask.long().sum(dim=1) - 1
            batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
            pooled = hidden_states[batch_idx, lengths, :]

        pooled = pooled.to(device=self.score.weight.device, dtype=self.score.weight.dtype)
        logits = self.score(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss": loss, "logits": logits}

In [71]:
base_lm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model = CausalLMForSequenceClassification(
    base_lm=base_lm,
    num_labels=2,
    pad_token_id=tokenizer.pad_token_id,
)
model = model.to(next(base_lm.parameters()).device)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = model.to("cuda")



print("Model loaded:", MODEL_NAME)
print("Model loaded (wrapped CausalLM):", MODEL_NAME)
print("Pad token id:", model.config.pad_token_id)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded: meta-llama/Llama-3.2-1B-Instruct
Model loaded (wrapped CausalLM): meta-llama/Llama-3.2-1B-Instruct
Pad token id: 128009


In [72]:
# changed this cell with the previous one
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=2,
#     id2label=ID_TO_LABEL,
#     label2id=LABEL_TO_ID,
#     trust_remote_code=True,
#     dtype=torch.float32,   # Keep trainable classifier/adapter weights in float32
# )

# model.config.pad_token_id = tokenizer.pad_token_id
# model.config.use_cache = False

# model = model.to("cuda")

# print("Model loaded:", MODEL_NAME)

# 14. Apply PEFT / LoRA

In [73]:
if not USE_LORA:
    raise ValueError("PEFT/LoRA is required for this project. Set USE_LORA = True.")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["score"],
    bias="none",
)

model = get_peft_model(model, lora_config)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

model.print_trainable_parameters()

trainable_dtype_counts = {}
unexpected_trainable_dtypes = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        # print(name, param.dtype)
        trainable_dtype_counts[str(param.dtype)] = (
            trainable_dtype_counts.get(str(param.dtype), 0) + param.numel()
        )

        if param.dtype != torch.float32:
            unexpected_trainable_dtypes[name] = str(param.dtype)

print("Trainable parameter dtype counts:")
print(trainable_dtype_counts)

if unexpected_trainable_dtypes:
    examples = list(unexpected_trainable_dtypes.items())[:10]
    raise RuntimeError(
        "Unexpected trainable parameter dtype(s). "
        "The model should be loaded with dtype=torch.float32 for this T4 fp16 run. "
        f"Examples: {examples}"
    )


trainable params: 856,066 || all params: 1,236,674,564 || trainable%: 0.0692
Trainable parameter dtype counts:
{'torch.float32': 856066}


# 15. Define Metrics

In [74]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=POSITIVE_LABEL_ID,
        zero_division=0,
    )

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }


# 16. Define Weighted Loss Trainer

In [75]:
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)

        if class_weights is None:
            raise ValueError("class_weights must be provided.")

        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs["logits"]

        # The loss weights must use the same device and dtype as the model logits.
        weights = self.class_weights.to(
            device=logits.device,
            dtype=logits.dtype,
        )

        labels = labels.to(
            device=logits.device,
            dtype=torch.long,
        )

        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        if return_outputs:
            return loss, outputs

        return loss


# 17. Memory Logging Callback

In [76]:
class TrainingProgressCallback(TrainerCallback):
    """
    Explicit Kaggle Save Version logging.

    Kaggle Save Version logs preserve print statements more reliably than
    tqdm progress bars, so this callback prints training, validation,
    checkpoint, and memory progress with flush=True.
    """

    def __init__(self, log_every_n_steps: int = LOGGING_STEPS):
        self.log_every_n_steps = log_every_n_steps

    def _memory_snapshot(self):
        process = psutil.Process(os.getpid())

        snapshot = {
            "cpu_ram_gb": round(process.memory_info().rss / (1024 ** 3), 3),
        }

        if torch.cuda.is_available():
            snapshot.update(
                {
                    "gpu_allocated_gb": round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
                    "gpu_reserved_gb": round(torch.cuda.memory_reserved() / (1024 ** 3), 3),
                    "gpu_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
                }
            )

        return snapshot

    def on_train_begin(self, args, state, control, **kwargs):
        print("\nTRAINING STARTED", flush=True)
        print(f"Total epochs: {args.num_train_epochs}", flush=True)
        print(f"Logging every {self.log_every_n_steps} steps", flush=True)

        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

        print("Initial memory:", self._memory_snapshot(), flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        step = state.global_step

        if step == 0:
            return

        if step % self.log_every_n_steps != 0:
            return

        message = {
            "step": int(step),
            "epoch": round(float(state.epoch), 4) if state.epoch is not None else None,
        }

        for key in ["loss", "learning_rate", "grad_norm"]:
            if key in logs:
                message[key] = logs[key]

        message.update(self._memory_snapshot())

        print("TRAIN LOG:", message, flush=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        message = {
            "step": int(state.global_step),
            "epoch": round(float(state.epoch), 4) if state.epoch is not None else None,
            "eval_loss": metrics.get("eval_loss"),
            "eval_accuracy": metrics.get("eval_accuracy"),
            "eval_f1": metrics.get("eval_f1"),
            "eval_f1_macro": metrics.get("eval_f1_macro"),
        }

        print("\nVALIDATION EVALUATION FINISHED", flush=True)
        print(message, flush=True)

    def on_save(self, args, state, control, **kwargs):
        print(f"\nCHECKPOINT SAVED at step {state.global_step}", flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print("\nTRAINING FINISHED", flush=True)
        print(f"Final step: {state.global_step}", flush=True)
        print("Final memory:", self._memory_snapshot(), flush=True)


# 17. Training Arguments

In [77]:
# This notebook assumes a Kaggle Tesla T4 GPU.
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Please enable a Kaggle T4 GPU accelerator.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name, flush=True)

if "T4" not in gpu_name:
    print("Warning: This notebook is configured for T4, but detected:", gpu_name, flush=True)

USE_FP16 = True

# Some Transformers versions use eval_strategy, others use evaluation_strategy.
training_args_signature = inspect.signature(TrainingArguments.__init__).parameters

# Compute warmup_steps manually instead of using warmup_ratio.
effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = math.ceil(len(train_df) / effective_batch_size)
total_training_steps = math.ceil(steps_per_epoch * NUM_EPOCHS)
WARMUP_STEPS = int(total_training_steps * WARMUP_RATIO)

print("Effective batch size:", effective_batch_size, flush=True)
print("Steps per epoch:", steps_per_epoch, flush=True)
print("Total training steps:", total_training_steps, flush=True)
print("Warmup steps:", WARMUP_STEPS, flush=True)
print("Logging steps:", LOGGING_STEPS, flush=True)
print("Using fp16:", USE_FP16, flush=True)

args_kwargs = {
    "output_dir": str(OUTPUT_DIR),

    # Core training settings
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "max_grad_norm": MAX_GRAD_NORM,

    # Logging/checkpointing
    "logging_steps": LOGGING_STEPS,
    "logging_first_step": True,
    "disable_tqdm": True,
    "save_strategy": "epoch",
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "greater_is_better": True,
    "report_to": "none",

    # T4 settings
    "fp16": USE_FP16,
    "dataloader_pin_memory": True,
    "remove_unused_columns": False,
    "seed": SEED,
}

if "eval_strategy" in training_args_signature:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_kwargs)

training_args


GPU: Tesla T4
Effective batch size: 8
Steps per epoch: 1173
Total training steps: 3519
Warmup steps: 105
Logging steps: 20
Using fp16: True


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=True,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False

# 18. Create Trainer

In [78]:
progress_callback = TrainingProgressCallback(log_every_n_steps=LOGGING_STEPS)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "class_weights": class_weights,
    "callbacks": [progress_callback],
}

# Newer Transformers versions use processing_class instead of tokenizer.
trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer

trainer = WeightedLossTrainer(**trainer_kwargs)
trainer


# 19. Train Model

In [79]:
dtype_counts = {}
unexpected_trainable_dtypes = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype_counts[str(param.dtype)] = dtype_counts.get(str(param.dtype), 0) + param.numel()

        if param.dtype != torch.float32:
            unexpected_trainable_dtypes[name] = str(param.dtype)

print("Trainable parameter dtype counts:", dtype_counts, flush=True)

if unexpected_trainable_dtypes:
    examples = list(unexpected_trainable_dtypes.items())[:10]
    raise RuntimeError(
        "Unexpected trainable parameter dtype(s) before training. "
        f"Examples: {examples}"
    )

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

    print("\nGPU MEMORY BEFORE TRAINING", flush=True)
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
    print("Initial allocated GB:", round(torch.cuda.memory_allocated() / (1024 ** 3), 3), flush=True)
    print("Initial reserved GB:", round(torch.cuda.memory_reserved() / (1024 ** 3), 3), flush=True)
else:
    print("No GPU detected.", flush=True)


Trainable parameter dtype counts: {'torch.float32': 856066}

GPU MEMORY BEFORE TRAINING
GPU: Tesla T4
Initial allocated GB: 6.957
Initial reserved GB: 12.182


In [80]:
print("\nStarting trainer.train()...", flush=True)

train_result = trainer.train()

print("\ntrainer.train() completed.", flush=True)

train_metrics = train_result.metrics
train_metrics_path = METRICS_OUTPUT_DIR / "train_runtime_metrics.json"

with open(train_metrics_path, "w") as f:
    json.dump(train_metrics, f, indent=2)

print("\nTraining runtime metrics:", flush=True)
print(train_metrics, flush=True)
print("Saved training runtime metrics to:", train_metrics_path, flush=True)

trainer_log_history_df = pd.DataFrame(trainer.state.log_history)

trainer_log_history_path = METRICS_OUTPUT_DIR / "trainer_log_history.csv"
trainer_log_history_df.to_csv(trainer_log_history_path, index=False)

print("\nSaved trainer log history to:", trainer_log_history_path, flush=True)
print(trainer_log_history_df.tail(), flush=True)

training_progress_df = trainer_log_history_df[
    trainer_log_history_df["loss"].notna()
].copy()

training_progress_path = METRICS_OUTPUT_DIR / "training_progress.csv"
training_progress_df.to_csv(training_progress_path, index=False)

print("\nSaved training progress to:", training_progress_path, flush=True)

if not training_progress_df.empty:
    progress_columns = [
        col for col in ["epoch", "step", "loss", "learning_rate", "grad_norm"]
        if col in training_progress_df.columns
    ]
    print(training_progress_df[progress_columns].tail(), flush=True)

train_metrics



Starting trainer.train()...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



TRAINING STARTED
Total epochs: 3
Logging every 20 steps
GPU: Tesla T4
Initial memory: {'cpu_ram_gb': 3.615, 'gpu_allocated_gb': 6.957, 'gpu_reserved_gb': 7.117, 'gpu_peak_allocated_gb': 6.957}
{'loss': '8.404', 'grad_norm': '353.7', 'learning_rate': '0', 'epoch': '0.001705'}
TRAIN LOG: {'step': 20, 'epoch': 0.0341, 'loss': 4.626239174290707, 'learning_rate': 3.619047619047619e-05, 'grad_norm': 93.55741882324219, 'cpu_ram_gb': 3.615, 'gpu_allocated_gb': 6.963, 'gpu_reserved_gb': 12.963, 'gpu_peak_allocated_gb': 11.584}
{'loss': '4.626', 'grad_norm': '93.56', 'learning_rate': '3.619e-05', 'epoch': '0.0341'}
TRAIN LOG: {'step': 40, 'epoch': 0.0682, 'loss': 2.944173049926758, 'learning_rate': 7.428571428571429e-05, 'grad_norm': 93.95146942138672, 'cpu_ram_gb': 3.615, 'gpu_allocated_gb': 6.963, 'gpu_reserved_gb': 12.963, 'gpu_peak_allocated_gb': 11.584}
{'loss': '2.944', 'grad_norm': '93.95', 'learning_rate': '7.429e-05', 'epoch': '0.0682'}
TRAIN LOG: {'step': 60, 'epoch': 0.1023, 'loss': 

{'train_runtime': 7170.5461,
 'train_samples_per_second': 3.926,
 'train_steps_per_second': 0.246,
 'train_loss': 2.040571437512928,
 'epoch': 3.0}

# 20. Evaluate on Validation Set

In [81]:
# Evaluation helpers used for train, validation, and test.
# The output format is intentionally the same for every split.

METRIC_COLUMNS_PRESENT = [
    "target_smell",
    "rows",
    "present",
    "absent",
    "accuracy",
    "precision",
    "recall",
    "f1",
]

METRIC_COLUMNS_MACRO = [
    "target_smell",
    "rows",
    "present",
    "absent",
    "macro_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
]

SMELL_ORDER = ["ALL", "blob", "data class", "feature envy", "long method"]


def load_aligned_metadata(path: Path, current_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Load only target_smell and label_id, aligned to the current split rows."""
    meta_df = pd.read_csv(path, usecols=["target_smell", "label_id"])

    if len(meta_df) == len(current_df):
        return meta_df.reset_index(drop=True)

    if not DEBUG_RUN:
        raise ValueError(
            f"{split_name} metadata row count does not match the model dataframe. "
            f"metadata={len(meta_df)}, model_df={len(current_df)}"
        )

    # Recreate the same debug indices used earlier, without keeping the full
    # 20-column dataset in memory.
    label_only_df = pd.read_csv(path, usecols=["label_id"])
    debug_indices = stratified_debug_indices(label_only_df, len(current_df), SEED)
    meta_df = meta_df.loc[debug_indices].reset_index(drop=True)

    if len(meta_df) != len(current_df):
        raise ValueError(
            f"{split_name} metadata is still misaligned after debug subsetting."
        )

    return meta_df


def compute_group_metrics(y_true, y_pred) -> dict:
    """Compute present-class metrics and macro metrics for one group."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=POSITIVE_LABEL_ID,
        zero_division=0,
    )

    per_class_precision, per_class_recall, per_class_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    macro_precision = float(np.mean(per_class_precision))
    macro_recall = float(np.mean(per_class_recall))
    macro_f1 = float(np.mean(per_class_f1))

    # macro_accuracy is balanced accuracy:
    # average recall across absent and present.
    macro_accuracy = macro_recall

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "macro_accuracy": float(macro_accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }


def build_metric_tables(eval_df: pd.DataFrame):
    """Build the two agreed non-redundant tables for one split."""
    rows = []

    groups = [("ALL", eval_df)]

    for smell in SMELL_ORDER[1:]:
        group = eval_df[eval_df["target_smell"] == smell]

        if not group.empty:
            groups.append((smell, group))

    for target_smell, group in groups:
        metrics = compute_group_metrics(
            group["true_label_id"].values,
            group["pred_label_id"].values,
        )

        rows.append(
            {
                "target_smell": target_smell,
                "rows": int(len(group)),
                "present": int((group["true_label_id"] == 1).sum()),
                "absent": int((group["true_label_id"] == 0).sum()),
                **metrics,
            }
        )

    metrics_df = pd.DataFrame(rows)

    present_class_metrics = metrics_df[METRIC_COLUMNS_PRESENT].copy()
    macro_metrics = metrics_df[METRIC_COLUMNS_MACRO].copy()

    return present_class_metrics, macro_metrics


def evaluate_split(split_name: str, tokenized_dataset, metadata_df: pd.DataFrame):
    """Run predictions and return predictions plus the two metric tables."""
    output = trainer.predict(tokenized_dataset)

    logits = output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    true_ids = output.label_ids
    pred_ids = np.argmax(logits, axis=-1)
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    eval_df = metadata_df.copy().reset_index(drop=True)

    if len(eval_df) != len(true_ids):
        raise ValueError(
            f"{split_name} metadata and predictions have different row counts. "
            f"metadata={len(eval_df)}, predictions={len(true_ids)}"
        )

    if not np.array_equal(eval_df["label_id"].values, true_ids):
        raise ValueError(f"{split_name} metadata labels do not align with model labels.")

    eval_df["true_label_id"] = true_ids
    eval_df["pred_label_id"] = pred_ids
    eval_df["true_label"] = eval_df["true_label_id"].map(ID_TO_LABEL)
    eval_df["pred_label"] = eval_df["pred_label_id"].map(ID_TO_LABEL)
    eval_df["absent_probability"] = probabilities[:, 0]
    eval_df["present_probability"] = probabilities[:, 1]

    present_class_metrics, macro_metrics = build_metric_tables(eval_df)

    return eval_df, present_class_metrics, macro_metrics


# 21. Evaluate on Test Set

In [82]:
print("\nLoading metadata for evaluation...", flush=True)

train_meta_df = load_aligned_metadata(train_path, train_df, "train")
val_meta_df = load_aligned_metadata(val_path, val_df, "validation")
test_meta_df = load_aligned_metadata(test_path, test_df, "test")

print("Metadata loaded.", flush=True)

print("\nEvaluating TRAIN split...", flush=True)
train_predictions_df, train_present_metrics, train_macro_metrics = evaluate_split(
    "train",
    tokenized_train,
    train_meta_df,
)
print("TRAIN evaluation completed.", flush=True)

print("\nEvaluating VALIDATION split...", flush=True)
val_predictions_df, val_present_metrics, val_macro_metrics = evaluate_split(
    "validation",
    tokenized_val,
    val_meta_df,
)
print("VALIDATION evaluation completed.", flush=True)

print("\nEvaluating TEST split...", flush=True)
test_predictions_df, test_present_metrics, test_macro_metrics = evaluate_split(
    "test",
    tokenized_test,
    test_meta_df,
)
print("TEST evaluation completed.", flush=True)

# Keep this variable for the final summary cell.
# It contains the TEST / ALL present-class metrics.
test_metrics = (
    test_present_metrics[test_present_metrics["target_smell"] == "ALL"]
    .iloc[0]
    .drop(labels=["target_smell"])
    .to_dict()
)

print("\nTEST / ALL present-class metrics:", flush=True)
print(test_metrics, flush=True)

test_metrics



Loading metadata for evaluation...
Metadata loaded.

Evaluating TRAIN split...
TRAIN evaluation completed.

Evaluating VALIDATION split...
VALIDATION evaluation completed.

Evaluating TEST split...
TEST evaluation completed.

TEST / ALL present-class metrics:
{'rows': 2027, 'present': 462, 'absent': 1565, 'accuracy': 0.7538233843117909, 'precision': 0.46322067594433397, 'recall': 0.5043290043290043, 'f1': 0.48290155440414506}


{'rows': 2027,
 'present': 462,
 'absent': 1565,
 'accuracy': 0.7538233843117909,
 'precision': 0.46322067594433397,
 'recall': 0.5043290043290043,
 'f1': 0.48290155440414506}

# 22. Per-Smell Evaluation

In [83]:
# Display and save the agreed metric tables.
# Each split has:
#   1. Present Class Metrics
#   2. Macro Metrics

split_results = {
    "train": {
        "present": train_present_metrics,
        "macro": train_macro_metrics,
        "predictions": train_predictions_df,
    },
    "validation": {
        "present": val_present_metrics,
        "macro": val_macro_metrics,
        "predictions": val_predictions_df,
    },
    "test": {
        "present": test_present_metrics,
        "macro": test_macro_metrics,
        "predictions": test_predictions_df,
    },
}

for split_name, result in split_results.items():
    present_metrics = result["present"]
    macro_metrics = result["macro"]
    predictions = result["predictions"]

    present_metrics_path = METRICS_OUTPUT_DIR / f"{split_name}_present_class_metrics.csv"
    macro_metrics_path = METRICS_OUTPUT_DIR / f"{split_name}_macro_metrics.csv"
    predictions_path = PREDICTIONS_OUTPUT_DIR / f"{split_name}_predictions.csv"

    present_metrics.to_csv(present_metrics_path, index=False)
    macro_metrics.to_csv(macro_metrics_path, index=False)
    predictions.to_csv(predictions_path, index=False)

    print(f"\n{split_name.upper()} - Present Class Metrics", flush=True)
    print(present_metrics.to_string(index=False), flush=True)
    display(present_metrics)

    print(f"\n{split_name.upper()} - Macro Metrics", flush=True)
    print(macro_metrics.to_string(index=False), flush=True)
    display(macro_metrics)

    print("Saved:", present_metrics_path, flush=True)
    print("Saved:", macro_metrics_path, flush=True)
    print("Saved:", predictions_path, flush=True)

# Also save one compact JSON file with the overall ALL-row metrics for quick lookup.
overall_metrics = {
    split_name: {
        "present_class_metrics": result["present"][
            result["present"]["target_smell"] == "ALL"
        ]
        .iloc[0]
        .to_dict(),
        "macro_metrics": result["macro"][
            result["macro"]["target_smell"] == "ALL"
        ]
        .iloc[0]
        .to_dict(),
    }
    for split_name, result in split_results.items()
}

overall_metrics_path = METRICS_OUTPUT_DIR / "overall_metrics.json"
with open(overall_metrics_path, "w") as f:
    json.dump(overall_metrics, f, indent=2)

print("\nSaved overall metrics to:", overall_metrics_path, flush=True)



TRAIN - Present Class Metrics
target_smell  rows  present  absent  accuracy  precision   recall       f1
         ALL  9383     2051    7332  0.859000   0.643533 0.795709 0.711576
        blob  2572      619    1953  0.850311   0.654354 0.801292 0.720407
  data class  2577      673    1904  0.845557   0.651599 0.878158 0.748101
feature envy  2110      280    1830  0.836967   0.388112 0.396429 0.392226
 long method  2124      479    1645  0.907721   0.741880 0.906054 0.815789


,target_smell,rows,present,absent,accuracy,precision,recall,f1
0,ALL,9383,2051,7332,0.859000,0.643533,0.795709,0.711576
1,blob,2572,619,1953,0.850311,0.654354,0.801292,0.720407
2,data class,2577,673,1904,0.845557,0.651599,0.878158,0.748101
3,feature envy,2110,280,1830,0.836967,0.388112,0.396429,0.392226
4,long method,2124,479,1645,0.907721,0.741880,0.906054,0.815789



TRAIN - Macro Metrics
target_smell  rows  present  absent  macro_accuracy  macro_precision  macro_recall  macro_f1
         ALL  9383     2051    7332        0.836207         0.791169      0.836207  0.809135
        blob  2572      619    1953        0.833570         0.793274      0.833570  0.809102
  data class  2577      673    1904        0.856096         0.801248      0.856096  0.818371
feature envy  2110      280    1830        0.650400         0.647729      0.650400  0.649041
 long method  2124      479    1645        0.907130         0.856320      0.907130  0.877116


,target_smell,rows,present,absent,macro_accuracy,macro_precision,macro_recall,macro_f1
0,ALL,9383,2051,7332,0.836207,0.791169,0.836207,0.809135
1,blob,2572,619,1953,0.833570,0.793274,0.833570,0.809102
2,data class,2577,673,1904,0.856096,0.801248,0.856096,0.818371
3,feature envy,2110,280,1830,0.650400,0.647729,0.650400,0.649041
4,long method,2124,479,1645,0.907130,0.856320,0.907130,0.877116


Saved: /kaggle/working/outputs/training/metrics/train_present_class_metrics.csv
Saved: /kaggle/working/outputs/training/metrics/train_macro_metrics.csv
Saved: /kaggle/working/outputs/training/predictions/train_predictions.csv

VALIDATION - Present Class Metrics
target_smell  rows  present  absent  accuracy  precision   recall       f1
         ALL  2034      454    1580  0.771386   0.489109 0.544053 0.515120
        blob   558      122     436  0.754480   0.419355 0.319672 0.362791
  data class   557      170     387  0.664273   0.464730 0.658824 0.545012
feature envy   458       62     396  0.831878   0.325581 0.225806 0.266667
 long method   461      100     361  0.861171   0.640625 0.820000 0.719298


,target_smell,rows,present,absent,accuracy,precision,recall,f1
0,ALL,2034,454,1580,0.771386,0.489109,0.544053,0.515120
1,blob,558,122,436,0.754480,0.419355,0.319672,0.362791
2,data class,557,170,387,0.664273,0.464730,0.658824,0.545012
3,feature envy,458,62,396,0.831878,0.325581,0.225806,0.266667
4,long method,461,100,361,0.861171,0.640625,0.820000,0.719298


... (3 outputs left out)

,target_smell,rows,present,absent,accuracy,precision,recall,f1
0,ALL,2027,462,1565,0.753823,0.463221,0.504329,0.482902
1,blob,545,133,412,0.715596,0.409836,0.375940,0.392157
2,data class,543,135,408,0.662983,0.368132,0.496296,0.422713
3,feature envy,468,70,398,0.794872,0.314286,0.314286,0.314286
4,long method,471,124,347,0.861996,0.728682,0.758065,0.743083



TEST - Macro Metrics
target_smell  rows  present  absent  macro_accuracy  macro_precision  macro_recall  macro_f1
         ALL  2027      462    1565        0.665903         0.656479      0.665903  0.660680
        blob   545      133     412        0.600591         0.606809      0.600591  0.603264
  data class   543      135     408        0.607217         0.589883      0.607217  0.592371
feature envy   468       70     398        0.596841         0.596841      0.596841  0.596841
 long method   471      124     347        0.828600         0.820481      0.828600  0.824372


,target_smell,rows,present,absent,macro_accuracy,macro_precision,macro_recall,macro_f1
0,ALL,2027,462,1565,0.665903,0.656479,0.665903,0.660680
1,blob,545,133,412,0.600591,0.606809,0.600591,0.603264
2,data class,543,135,408,0.607217,0.589883,0.607217,0.592371
3,feature envy,468,70,398,0.596841,0.596841,0.596841,0.596841
4,long method,471,124,347,0.828600,0.820481,0.828600,0.824372


Saved: /kaggle/working/outputs/training/metrics/test_present_class_metrics.csv
Saved: /kaggle/working/outputs/training/metrics/test_macro_metrics.csv
Saved: /kaggle/working/outputs/training/predictions/test_predictions.csv

Saved overall metrics to: /kaggle/working/outputs/training/metrics/overall_metrics.json


# 23. Save Model and Tokenizer

In [84]:
# For PEFT training, this saves the LoRA adapter and classifier-related weights.
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

print("Saved model to:", MODEL_OUTPUT_DIR)

Saved model to: /kaggle/working/outputs/training/model


# 24. Manual Prediction Test

In [85]:
def predict_text(text: str) -> dict:
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs["logits"]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]

    pred_id = int(np.argmax(probs))

    return {
        "prediction_id": pred_id,
        "prediction_label": ID_TO_LABEL[pred_id],
        "absent_probability": float(probs[0]),
        "present_probability": float(probs[1]),
    }


example = test_df.iloc[0]

print("True label_id:", int(example["label_id"]))
print("True label:", ID_TO_LABEL[int(example["label_id"])])

predict_text(example["text"])

True label_id: 0
True label: absent


{'prediction_id': 0,
 'prediction_label': 'absent',
 'absent_probability': 0.676778256893158,
 'present_probability': 0.32322177290916443}

# 25. Final Summary

In [86]:
summary = {
    "model_name": MODEL_NAME,
    "use_lora": USE_LORA,
    "max_length": MAX_LENGTH,
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "training_parameters": {
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": WARMUP_STEPS,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "max_grad_norm": MAX_GRAD_NORM,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "logging_steps": LOGGING_STEPS,
        "fp16": True,
    },
    "lora_parameters": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
    },
    "class_weights": {
        "absent": float(class_weights[0]),
        "present": float(class_weights[1]),
    },
    "test_metrics": test_metrics,
    "model_output_dir": str(MODEL_OUTPUT_DIR),
    "metrics_output_dir": str(METRICS_OUTPUT_DIR),
    "predictions_output_dir": str(PREDICTIONS_OUTPUT_DIR),
}

with open(OUTPUT_DIR / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary


{'model_name': 'meta-llama/Llama-3.2-1B-Instruct',
 'use_lora': True,
 'max_length': 512,
 'train_rows': 9383,
 'validation_rows': 2034,
 'test_rows': 2027,
 'training_parameters': {'num_epochs': 3,
  'learning_rate': 0.0002,
  'train_batch_size': 2,
  'eval_batch_size': 2,
  'gradient_accumulation_steps': 4,
  'effective_batch_size': 8,
  'weight_decay': 0.01,
  'warmup_ratio': 0.03,
  'warmup_steps': 105,
  'lr_scheduler_type': 'linear',
  'max_grad_norm': 1.0,
  'save_total_limit': 1,
  'logging_steps': 20,
  'fp16': True},
 'lora_parameters': {'r': 8,
  'lora_alpha': 16,
  'lora_dropout': 0.05,
  'target_modules': ['q_proj', 'v_proj']},
 'class_weights': {'absent': 0.6398663520812988,
  'present': 2.2874207496643066},
 'test_metrics': {'rows': 2027,
  'present': 462,
  'absent': 1565,
  'accuracy': 0.7538233843117909,
  'precision': 0.46322067594433397,
  'recall': 0.5043290043290043,
  'f1': 0.48290155440414506},
 'model_output_dir': '/kaggle/working/outputs/training/model',
 'met